This notebook applies requirement def and attribute override to the heating subsystem; after running it you can see how the same two constructs from Chapter 2 recur at the second level of decomposition.

Chapter 2 introduced requirement def and attribute override at the top-level `Toaster`. Chapter 6 applies the same pattern one level down: the `Heater` part definition now has its own requirement (`HeatingReq`) and a variant with an overridden `power` attribute.

This is the pedagogical core of recursive decomposition: the pattern does not change as you go deeper. Each level has a formal specification, candidate variants, and satisfaction claims.

In [ ]:
from pathlib import Path
import opensysml
from toaster.report import format_diagnostics

conn = opensysml.connect(version="v0.9.0")
source = Path("../../models/ch06-cumulative.sysml").read_text()
print(source)
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"

The `ch06-cumulative.sysml` file adds a second-level requirement: `HeatingReq` constrains `heater.power >= 600.0` W on the `Heater` sub-component. A `HeatingAssembly` decomposition adds `ResistanceCoil` and `PowerWire` sub-parts. Two candidate heaters — `efficient` (800 W) and `weak` (400 W) — exercise the new requirement using the same satisfy-assertion pattern from Chapter 3, applied one level down in the hierarchy.

In [ ]:
# Negative control: overriding an attribute that does not exist in the parent type
# raises "unresolved reference" — the override target must name a declared attribute.
bad_source = """
package BadSubsys {
    private import ScalarValues::*;
    part def Heater { attribute power : Real default = 800.0; }
    part def BadVariant :> Heater {
        attribute :>> nonExistentAttr = 500.0;
    }
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok
print(f"Neg control diagnostics: {bad.diagnostics[0].message!r}")

In [ ]:
# Navigate to the subsystem requirement using find/get (Ch5 nav op)
heating_req = model.find("ToasterDemo::HeatingReq")
print(f"HeatingReq: id={heating_req.id!r}, kind={heating_req.kind!r}")

efficient = model.find("ToasterDemo::efficient")
print(f"efficient variant: id={efficient.id!r}, kind={efficient.kind!r}")

weak = model.find("ToasterDemo::weak")
print(f"weak variant: id={weak.id!r}, kind={weak.kind!r}")

The SysML v2 requirement def and attribute override constructs (A-F) are applied to the `Heater` subsystem and parsed by OpenSysML (O-S); `model.find()` confirms the subsystem requirement and both variants are present as named model elements (E).

Try the chapter exercise in `exercises/ch06/exercise.ipynb`: add a `BrewReq` requirement for your coffee maker's `BrewUnit`, create an `overTemp` variant with an overridden `waterTemp` attribute, and confirm both the requirement and the variant appear with `model.find()`.